# GraphRetro 源码拆解与预测流程

这一份 notebook 是整套教材里最接近"读源码"的部分。目标不是把源码一行不差地背下来，而是帮助你建立一个清晰的问题框架：

1. GraphRetro 为什么要分成两个阶段？
2. 每个阶段分别在预测什么？
3. 一次完整的 retrosynthesis 推理，在代码里是怎样流动的？

建议始终抓住一条主线：

**product 先被编码成图，再预测 bond edit（断键/改键），得到 synthons，最后给每个 synthon 补上 leaving group 得到 reactants。**

---

GraphRetro 的核心思路可以概括为：

$$
P(R \mid P) = \sum_{e} P(e \mid P) \cdot P(\text{LG} \mid S(e, P))
$$

其中：

- $P(e \mid P)$：**Stage 1** — 给定产物 P，预测键编辑 (bond edit) e
- $S(e, P)$：根据编辑 e 把产物 P 切成中间碎片 (synthons)
- $P(\text{LG} \mid S)$：**Stage 2** — 给定 synthons，预测需要补上的离去基团 (leaving group)


In [1]:
import os
import sys
from pathlib import Path


def find_project_root(start=None):
    try:
        here = Path(start or os.getcwd()).resolve()
    except (FileNotFoundError, OSError):
        # 内核工作目录可能已失效，从 notebook 文件路径回退
        _nb = globals().get("__vsc_ipynb_file__")
        if _nb:
            here = Path(_nb).resolve().parent
        else:
            raise
    if here.is_file():
        here = here.parent
    for candidate in [here, *here.parents]:
        if (candidate / "teaching_demos").exists() and (candidate / "source_repos").exists():
            return candidate
    raise FileNotFoundError("无法定位项目根目录")


PROJECT_ROOT = find_project_root('.')
TUTORIAL_DIR = PROJECT_ROOT / "teaching_demos/2.single_step_retro_tutorial/2.3.semi-template/2.3.2.graphretro"
DATA_DIR = TUTORIAL_DIR / "data"
SOURCE_DIR = PROJECT_ROOT / "source_repos/graphretro"

print(f"项目根目录: {PROJECT_ROOT}")
print(f"教程目录:   {TUTORIAL_DIR}")

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

torch.manual_seed(7)


def show_source_block(relative_path, start, end, title):
    """从源码仓库中读取指定行范围的代码并格式化展示。"""
    path = PROJECT_ROOT / relative_path
    lines = path.read_text(encoding="utf-8").splitlines()
    snippet = "\n".join(f"{line_no:4d} | {lines[line_no - 1]}" for line_no in range(start, end + 1))
    display(Markdown(f"### {title}\n`{relative_path}:{start}-{end}`"))
    display(Markdown(f"```python\n{snippet}\n```"))


SOURCE_BLOCKS = [
    {
        "阶段": "原子/键特征",
        "source_file": "source_repos/graphretro/seq_graph_retro/molgraph/mol_features.py",
        "symbol": "get_atom_features / get_bond_features",
        "lines": "1-200",
    },
    {
        "阶段": "图构建与batch",
        "source_file": "source_repos/graphretro/seq_graph_retro/data/collate_fns.py",
        "symbol": "pack_graph_feats",
        "lines": "30-174",
    },
    {
        "阶段": "GNN 编码器",
        "source_file": "source_repos/graphretro/seq_graph_retro/layers/encoder.py",
        "symbol": "WLNEncoder / GraphFeatEncoder / GTransEncoder",
        "lines": "122-385",
    },
    {
        "阶段": "消息传递 RNN",
        "source_file": "source_repos/graphretro/seq_graph_retro/layers/rnn.py",
        "symbol": "MPNLayer / GRU",
        "lines": "7-220",
    },
    {
        "阶段": "Stage 1: 键编辑预测",
        "source_file": "source_repos/graphretro/seq_graph_retro/models/core_edits/single_edit.py",
        "symbol": "SingleEdit._compute_edit_logits / predict",
        "lines": "137-370",
    },
    {
        "阶段": "Stage 2: 离去基团预测",
        "source_file": "source_repos/graphretro/seq_graph_retro/models/lg_edits/lg_ind_embed.py",
        "symbol": "LGIndEmbed._compute_lg_logits / predict",
        "lines": "92-247",
    },
    {
        "阶段": "化学工具: 编辑应用",
        "source_file": "source_repos/graphretro/seq_graph_retro/utils/chem.py",
        "symbol": "apply_edits_to_mol",
        "lines": "30-180",
    },
    {
        "阶段": "反应解析",
        "source_file": "source_repos/graphretro/seq_graph_retro/utils/parse.py",
        "symbol": "get_reaction_core / get_reaction_info",
        "lines": "74-248",
    },
]

项目根目录: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis
教程目录:   /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/teaching_demos/2.single_step_retro_tutorial/2.3.semi-template/2.3.2.graphretro


## 1. 先看源码地图

在阅读任何一个复杂的模型源码之前，最有效的策略是先建立一个**全局地图**，知道每个模块的职责和位置。

下表列出了 GraphRetro 的 8 个核心源码块，从最基础的特征定义到最终的推理流程：


In [2]:
display(pd.DataFrame(SOURCE_BLOCKS))

,阶段,source_file,symbol,lines
0,原子/键特征,source_repos/graphretro/seq_graph_retro/molgra...,get_atom_features / get_bond_features,1-200
1,图构建与batch,source_repos/graphretro/seq_graph_retro/data/c...,pack_graph_feats,30-174
2,GNN 编码器,source_repos/graphretro/seq_graph_retro/layers...,WLNEncoder / GraphFeatEncoder / GTransEncoder,122-385
3,消息传递 RNN,source_repos/graphretro/seq_graph_retro/layers...,MPNLayer / GRU,7-220
4,Stage 1: 键编辑预测,source_repos/graphretro/seq_graph_retro/models...,SingleEdit._compute_edit_logits / predict,137-370
5,Stage 2: 离去基团预测,source_repos/graphretro/seq_graph_retro/models...,LGIndEmbed._compute_lg_logits / predict,92-247
6,化学工具: 编辑应用,source_repos/graphretro/seq_graph_retro/utils/...,apply_edits_to_mol,30-180
7,反应解析,source_repos/graphretro/seq_graph_retro/utils/...,get_reaction_core / get_reaction_info,74-248


## 2. 分子特征工程：原子与键的数字化表达

GraphRetro 的一切计算都建立在原子和键的特征向量之上。理解这些特征的组成，是理解整个模型的第一步。

**核心常量：**
- `ATOM_FDIM = 87`：每个原子用 87 维向量表示
- `BOND_FDIM = 6`：每条键用 6 维向量表示
- `MAX_NB = 10`：每个原子最多考虑 10 个邻居
- `BOND_FLOATS = [0.0, 1.0, 2.0, 3.0, 1.5]`：所有可能的键级（无键、单键、双键、三键、芳香键）


In [3]:
show_source_block(
    "source_repos/graphretro/seq_graph_retro/molgraph/mol_features.py",
    1, 60,
    "原子与键特征常量定义"
)

### 原子与键特征常量定义
`source_repos/graphretro/seq_graph_retro/molgraph/mol_features.py:1-60`

```python
   1 | import numpy as np
   2 | from rdkit import Chem
   3 | from typing import Set, Any, List, Union
   4 | 
   5 | from seq_graph_retro.utils.chem import get_mol
   6 | 
   7 | idxfunc = lambda a : a.GetAtomMapNum() - 1
   8 | bond_idx_fn = lambda a, b, mol: mol.GetBondBetweenAtoms(a.GetIdx(), b.GetIdx()).GetIdx()
   9 | 
  10 | # Symbols for different atoms
  11 | ATOM_LIST = ['C', 'N', 'O', 'S', 'F', 'Si', 'P', 'Cl', 'Br', 'Mg', 'Na', 'Ca', 'Fe', \
  12 |     'As', 'Al', 'I', 'B', 'V', 'K', 'Tl', 'Yb', 'Sb', 'Sn', 'Ag', 'Pd', 'Co', 'Se', 'Ti', \
  13 |     'Zn', 'H', 'Li', 'Ge', 'Cu', 'Au', 'Ni', 'Cd', 'In', 'Mn', 'Zr', 'Cr', 'Pt', 'Hg', 'Pb', \
  14 |     'W', 'Ru', 'Nb', 'Re', 'Te', 'Rh', 'Ta', 'Tc', 'Ba', 'Bi', 'Hf', 'Mo', 'U', 'Sm', 'Os', 'Ir', \
  15 |     'Ce','Gd','Ga','Cs', '*', 'unk']
  16 | 
  17 | MAX_NB = 10
  18 | DEGREES = list(range(MAX_NB))
  19 | HYBRIDIZATION = [Chem.rdchem.HybridizationType.SP,
  20 |         Chem.rdchem.HybridizationType.SP2,
  21 |         Chem.rdchem.HybridizationType.SP3,
  22 |         Chem.rdchem.HybridizationType.SP3D,
  23 |         Chem.rdchem.HybridizationType.SP3D2]
  24 | 
  25 | FORMAL_CHARGE = [-1, -2, 1, 2, 0]
  26 | VALENCE = [0, 1, 2, 3, 4, 5, 6]
  27 | NUM_Hs = [0, 1, 3, 4, 5]
  28 | 
  29 | BOND_TYPES = [None, Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE, \
  30 |     Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]
  31 | BOND_FLOAT_TO_TYPE = {
  32 |     0.0: BOND_TYPES[0],
  33 |     1.0: BOND_TYPES[1],
  34 |     2.0: BOND_TYPES[2],
  35 |     3.0: BOND_TYPES[3],
  36 |     1.5: BOND_TYPES[4],
  37 | }
  38 | 
  39 | BOND_DELTAS = {-3: 0, -2: 1, -1.5: 2, -1: 3, -0.5: 4, 0: 5, 0.5: 6, 1: 7, 1.5: 8, 2:9, 3:10}
  40 | BOND_FLOATS = [0.0, 1.0, 2.0, 3.0, 1.5]
  41 | RXN_CLASSES = list(range(10))
  42 | 
  43 | ATOM_FDIM = len(ATOM_LIST) + len(DEGREES) + len(FORMAL_CHARGE) + len(HYBRIDIZATION) \
  44 |             + len(VALENCE) + len(NUM_Hs) + 1
  45 | BOND_FDIM = 6
  46 | BINARY_FDIM = 5 + BOND_FDIM
  47 | INVALID_BOND = -1
  48 | PATTERN_DIM = 389
  49 | 
  50 | def sanitize(mol, kekulize: bool = True) -> Chem.Mol:
  51 |     """Sanitize mol.
  52 |     Parameters
  53 |     ----------
  54 |     mol: Chem.Mol
  55 |         Molecule to sanitize
  56 |     kekulize: bool
  57 |         Whether to kekulize the molecule
  58 |     """
  59 |     try:
  60 |         smiles = get_smiles(mol) if kekulize else Chem.MolToSmiles(mol)
```

In [4]:
# 用 RDKit 演示一个具体分子的特征提取
sys.path.insert(0, str(SOURCE_DIR))
from seq_graph_retro.molgraph.mol_features import (
    ATOM_FDIM, BOND_FDIM, ATOM_LIST, BOND_FLOATS, MAX_NB,
    get_atom_features, get_bond_features,
)
from rdkit import Chem

demo_smi = "c1ccc(O)cc1"  # 苯酚
mol = Chem.MolFromSmiles(demo_smi)

print(f"分子: {demo_smi}")
print(f"原子数: {mol.GetNumAtoms()},  键数: {mol.GetNumBonds()}")
print(f"ATOM_FDIM = {ATOM_FDIM},  BOND_FDIM = {BOND_FDIM}")
print(f"ATOM_LIST 长度: {len(ATOM_LIST)}")
print(f"BOND_FLOATS: {BOND_FLOATS}")
print()

# 展示第一个原子的特征
atom0 = mol.GetAtomWithIdx(0)
feat0 = np.array(get_atom_features(atom0))
print(f"原子 0 ({atom0.GetSymbol()}): 特征向量形状 = {feat0.shape}")

# 展示所有原子特征
atom_info = []
for atom in mol.GetAtoms():
    feat = np.array(get_atom_features(atom))
    atom_info.append({
        "idx": atom.GetIdx(),
        "symbol": atom.GetSymbol(),
        "degree": atom.GetTotalDegree(),
        "charge": atom.GetFormalCharge(),
        "aromatic": atom.GetIsAromatic(),
        "feat_dim": feat.shape[0],
        "feat_nonzero": int(feat.sum()),
    })
display(pd.DataFrame(atom_info))

# 展示键特征
bond_info = []
for bond in mol.GetBonds():
    feat = get_bond_features(bond)
    bond_info.append({
        "bond_idx": bond.GetIdx(),
        "begin": bond.GetBeginAtomIdx(),
        "end": bond.GetEndAtomIdx(),
        "type": str(bond.GetBondType()),
        "feat": feat.tolist(),
    })
display(pd.DataFrame(bond_info))

分子: c1ccc(O)cc1
原子数: 7,  键数: 7
ATOM_FDIM = 98,  BOND_FDIM = 6
ATOM_LIST 长度: 65
BOND_FLOATS: [0.0, 1.0, 2.0, 3.0, 1.5]

原子 0 (C): 特征向量形状 = (98,)


,idx,symbol,degree,charge,aromatic,feat_dim,feat_nonzero
0,0,C,3,0,True,98,7
1,1,C,3,0,True,98,7
2,2,C,3,0,True,98,7
3,3,C,3,0,True,98,7
4,4,O,2,0,False,98,6
5,5,C,3,0,True,98,7
6,6,C,3,0,True,98,7


,bond_idx,begin,end,type,feat
0,0,0,1,AROMATIC,"[0.0, 0.0, 0.0, 1.0, 1.0, 1.0]"
1,1,1,2,AROMATIC,"[0.0, 0.0, 0.0, 1.0, 1.0, 1.0]"
2,2,2,3,AROMATIC,"[0.0, 0.0, 0.0, 1.0, 1.0, 1.0]"
3,3,3,4,SINGLE,"[1.0, 0.0, 0.0, 0.0, 1.0, 0.0]"
4,4,3,5,AROMATIC,"[0.0, 0.0, 0.0, 1.0, 1.0, 1.0]"
5,5,5,6,AROMATIC,"[0.0, 0.0, 0.0, 1.0, 1.0, 1.0]"
6,6,6,0,AROMATIC,"[0.0, 0.0, 0.0, 1.0, 1.0, 1.0]"


从上面的输出可以看到两个关键事实：

1. **原子特征是 one-hot 拼接**：原子类型(67) + 度数(10) + 电荷(5) + 杂化(5) + 化合价(7) + 氢原子数(5) + 芳香性(1) = **87 维**
2. **键特征只有 6 维**：单键、双键、三键、芳香键、共轭、成环

这些数字化的特征就是后续 GNN 编码器的输入。


## 3. 图表示与 Batch 构建

GraphRetro 使用两种图表示：
- **无向图**（用于 WLNEncoder）：邻居原子索引
- **有向图**（用于 GraphFeatEncoder / GTransEncoder）：消息索引

`pack_graph_feats()` 负责把多个分子打包成一个大 batch 张量，这是理解模型输入的关键。


In [5]:
show_source_block(
    "source_repos/graphretro/seq_graph_retro/data/collate_fns.py",
    30, 100,
    "pack_graph_feats — 分子图的 Batch 构建"
)

### pack_graph_feats — 分子图的 Batch 构建
`source_repos/graphretro/seq_graph_retro/data/collate_fns.py:30-100`

```python
  30 | def pack_graph_feats(graph_batch: List[Any], directed: bool, use_rxn_class: bool = False,
  31 |                      return_graphs: bool = False) -> Tuple[torch.Tensor, List[Tuple[int]]]:
  32 |     """Prepare graph tensors.
  33 | 
  34 |     Parameters
  35 |     ----------
  36 |     graph_batch: List[Any],
  37 |         Batch of graph objects. Should have attributes G_dir, G_undir
  38 |     directed: bool,
  39 |         Whether to prepare tensors for directed message passing
  40 |     use_rxn_class: bool, default False,
  41 |         Whether to use reaction class as additional input
  42 |     return_graphs: bool, default False,
  43 |         Whether to return the graphs
  44 |     """
  45 |     if directed:
  46 |         fnode = [get_atom_features(Chem.Atom("*"), use_rxn_class=use_rxn_class, rxn_class=0)]
  47 |         fmess = [[0,0] + [0] * BOND_FDIM]
  48 |         agraph, bgraph = [[]], [[]]
  49 |         atoms_in_bonds = [[]]
  50 | 
  51 |         atom_scope, bond_scope = [], []
  52 |         edge_dict = {}
  53 |         all_G = []
  54 | 
  55 |         for bid, graph in enumerate(graph_batch):
  56 |             mol = graph.mol
  57 |             assert mol.GetNumAtoms() == len(graph.G_dir)
  58 |             atom_offset = len(fnode)
  59 |             bond_offset = len(atoms_in_bonds)
  60 | 
  61 |             bond_to_tuple = {bond.GetIdx(): tuple(sorted((bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())))
  62 |                              for bond in mol.GetBonds()}
  63 |             tuple_to_bond = {val: key for key, val in bond_to_tuple.items()}
  64 | 
  65 |             atom_scope.append(graph.update_atom_scope(atom_offset))
  66 |             bond_scope.append(graph.update_bond_scope(bond_offset))
  67 | 
  68 |             G = nx.convert_node_labels_to_integers(graph.G_dir, first_label=atom_offset)
  69 |             all_G.append(G)
  70 |             fnode.extend( [None for v in G.nodes] )
  71 | 
  72 |             for v, attr in G.nodes(data='label'):
  73 |                 G.nodes[v]['batch_id'] = bid
  74 |                 fnode[v] = get_atom_features(mol.GetAtomWithIdx(v-atom_offset),
  75 |                                              use_rxn_class=use_rxn_class,
  76 |                                              rxn_class=graph.rxn_class)
  77 |                 agraph.append([])
  78 | 
  79 |             bond_comp = [None for _ in range(mol.GetNumBonds())]
  80 |             for u, v, attr in G.edges(data='label'):
  81 |                 bond_feat = get_bond_features(mol.GetBondBetweenAtoms(u-atom_offset, v-atom_offset)).tolist()
  82 | 
  83 |                 bond = sorted([u, v])
  84 |                 mess_vec = [u, v] + bond_feat
  85 |                 if [v, u] not in bond_comp:
  86 |                     idx_to_add = tuple_to_bond[(u-atom_offset, v-atom_offset)]
  87 |                     bond_comp[idx_to_add] = [u, v]
  88 | 
  89 |                 fmess.append(mess_vec)
  90 |                 edge_dict[(u, v)] = eid = len(edge_dict) + 1
  91 |                 G[u][v]['mess_idx'] = eid
  92 |                 agraph[v].append(eid)
  93 |                 bgraph.append([])
  94 |             atoms_in_bonds.extend(bond_comp)
  95 | 
  96 |             for u, v in G.edges:
  97 |                 eid = edge_dict[(u, v)]
  98 |                 for w in G.predecessors(u):
  99 |                     if w == v: continue
 100 |                     bgraph[eid].append( edge_dict[(w, u)] )
```

In [6]:
from seq_graph_retro.molgraph.rxn_graphs import RxnElement, MultiElement
from seq_graph_retro.data.collate_fns import pack_graph_feats

# 构建产物的图表示
prod_smi = "c1ccc(O)cc1"
prod_mol = Chem.MolFromSmiles(prod_smi)
# 给原子编号（模拟 atom map）
for i, atom in enumerate(prod_mol.GetAtoms()):
    atom.SetAtomMapNum(i + 1)

prod_graph = RxnElement(mol=prod_mol)

# 无向图模式的 batch
graph_tensors_undir, scopes_undir = pack_graph_feats(
    [prod_graph], directed=False, use_rxn_class=False
)
# 有向图模式的 batch
graph_tensors_dir, scopes_dir = pack_graph_feats(
    [prod_graph], directed=True, use_rxn_class=False
)

print("=== 无向图模式 (WLNEncoder) ===")
names_undir = ["fnode", "fmess", "agraph", "bgraph", "atoms_in_bonds"]
for name, tensor in zip(names_undir, graph_tensors_undir):
    print(f"  {name:20s} shape = {tuple(tensor.shape)}")

print(f"  atom_scope = {scopes_undir[0]}")
print(f"  bond_scope = {scopes_undir[1]}")

print()
print("=== 有向图模式 (GraphFeatEncoder / GTransEncoder) ===")
names_dir = ["fnode", "fmess", "agraph", "bgraph", "atoms_in_bonds"]
for name, tensor in zip(names_dir, graph_tensors_dir):
    print(f"  {name:20s} shape = {tuple(tensor.shape)}")

print(f"  atom_scope = {scopes_dir[0]}")
print(f"  bond_scope = {scopes_dir[1]}")


=== 无向图模式 (WLNEncoder) ===
  fnode                shape = (8, 98)
  fmess                shape = (8, 6)
  agraph               shape = (8, 3)
  bgraph               shape = (8, 3)
  atoms_in_bonds       shape = (8, 2)
  atom_scope = [(1, 7)]
  bond_scope = [(1, 7)]

=== 有向图模式 (GraphFeatEncoder / GTransEncoder) ===
  fnode                shape = (8, 98)
  fmess                shape = (15, 8)
  agraph               shape = (8, 3)
  bgraph               shape = (15, 2)
  atoms_in_bonds       shape = (8, 2)
  atom_scope = [(1, 7)]
  bond_scope = [(1, 7)]


**关键理解点：**

- `fnode`：所有原子的特征矩阵 `[总原子数, 87]`，索引 0 是 padding
- `fmess`：消息特征 `[总消息数, 87+6]`（有向图模式）或键特征（无向图模式）
- `agraph`：原子→邻居 的邻接列表 `[总原子数, MAX_NB]`
- `bgraph`：消息→前驱消息 的邻接列表，用于消息传递的递推
- `scopes`：记录每个分子在大 batch 中的偏移量和长度，用于 pooling 回分子级别

这种 **padding + scope** 的设计模式在 GNN 框架中非常常见。


## 4. GNN 编码器：分子的向量化

GraphRetro 提供了三种编码器，它们的职责都一样：**把分子图编码成原子级别和分子级别的向量**。

| 编码器 | 消息传递机制 | 图类型 | 特点 |
|--------|-------------|--------|------|
| WLNEncoder | 邻域聚合 + 门控 | 无向 | 速度快，经典 WLN 风格 |
| GraphFeatEncoder | GRU/LSTM-based RNN | 有向 | 灵活性高 |
| GTransEncoder | Multi-head attention | 有向 | 可解释性好 |

我们以 **WLNEncoder** 为主线来讲解，它也是论文中最常用的默认选择。


In [7]:
show_source_block(
    "source_repos/graphretro/seq_graph_retro/layers/encoder.py",
    211, 320,
    "WLNEncoder — 核心图编码器"
)

### WLNEncoder — 核心图编码器
`source_repos/graphretro/seq_graph_retro/layers/encoder.py:211-320`

```python
 211 | class WLNEncoder(nn.Module):
 212 |     """
 213 |     WLNEncoder encodes molecules by using features of atoms and bonds, following
 214 |     the update rules as defined in the WLN Architecture.
 215 |     """
 216 |     def __init__(self,
 217 |                  n_atom_feat: int = ATOM_FDIM,
 218 |                  n_bond_feat: int = BOND_FDIM,
 219 |                  hsize: int = 64,
 220 |                  depth: int = 3,
 221 |                  bias: bool = False,
 222 |                  dropout_p: float = 0.15,
 223 |                  **kwargs) -> None:
 224 |         """
 225 |         Parameters
 226 |         ----------
 227 |         n_atom_feat: int, default ATOM_FDIM(87)
 228 |             Number of atom features
 229 |         n_bond_feat: int, default BOND_FDIM(6)
 230 |             Number of bond features
 231 |         hsize: int, default 64
 232 |             Size of the embeddings
 233 |         depth: int, default 3
 234 |             Depth of the WLN Graph Convolution
 235 |         bias: bool, default False
 236 |             Whether to use a bias term in the linear layers
 237 |         """
 238 |         super(WLNEncoder, self).__init__(**kwargs)
 239 |         self.n_atom_feat = n_atom_feat
 240 |         self.n_bond_feat = n_bond_feat
 241 |         self.hsize = hsize
 242 |         self.depth = depth
 243 |         self.bias = bias
 244 |         self.dropout_p = dropout_p
 245 |         self._build_layers()
 246 | 
 247 |     def _build_layers(self) -> None:
 248 |         """Builds the different layers associated."""
 249 |         self.atom_emb = nn.Linear(self.n_atom_feat, self.hsize, self.bias)
 250 |         self.bond_emb = nn.Linear(self.n_bond_feat, self.hsize, self.bias)
 251 | 
 252 |         self.U1 = nn.Linear(self.hsize, self.hsize, self.bias)
 253 |         self.U2 = nn.Linear(self.hsize, self.hsize, self.bias)
 254 |         self.V = nn.Linear(2 * self.hsize, self.hsize, self.bias)
 255 | 
 256 |         self.W0 = nn.Linear(self.hsize, self.hsize, self.bias)
 257 |         self.W1 = nn.Linear(self.hsize, self.hsize, self.bias)
 258 |         self.W2 = nn.Linear(self.hsize, self.hsize, self.bias)
 259 | 
 260 |         self.dropouts = []
 261 |         for i in range(self.depth):
 262 |             self.dropouts.append(nn.Dropout(p=self.dropout_p))
 263 |         self.dropouts = nn.ModuleList(self.dropouts)
 264 | 
 265 |     def forward(self, inputs: Tuple[torch.Tensor], scopes: Scope,
 266 |                 return_layers: bool = False) -> Tuple[torch.Tensor]:
 267 |         """Forward propagation step.
 268 | 
 269 |         Parameters
 270 |         ----------
 271 |         inputs: tuple of torch.tensors
 272 |             Graph tensors used as input for the WLNEmbedding
 273 |         scopes: Tuple[List]
 274 |             Scopes is composed of atom and bond scopes, which keep track of
 275 |             atom and bond indices for each molecule in the 2D feature list
 276 |         return layers: bool, default False,
 277 |             Whether to return the atom embeddings for every layer of graph convolutions
 278 |         """
 279 |         layers = []
 280 |         atom_feat, bond_feat, atom_graph, bond_graph, _ = inputs
 281 |         atom_scope, bond_scope = scopes
 282 |         bs = len(atom_scope)
 283 | 
 284 |         h_atom = self.atom_emb(atom_feat)
 285 |         h_bond = self.bond_emb(bond_feat)
 286 | 
 287 |         for i in range(self.depth):
 288 |             h_atom_nei = index_select_ND(h_atom, dim=0, index=atom_graph)
 289 |             h_bond_nei = index_select_ND(h_bond, dim=0, index=bond_graph)
 290 | 
 291 |             f_atom = self.W0(h_atom)
 292 |             f_bond_nei = self.W1(h_bond_nei)
 293 |             f_atom_nei = self.W2(h_atom_nei)
 294 | 
 295 |             c_atom = f_atom * torch.sum(f_atom_nei * f_bond_nei, dim=1)
 296 |             layers.append(c_atom)
 297 | 
 298 |             nei_label = F.relu(self.V(torch.cat([h_bond_nei, h_atom_nei], dim=-1)))
 299 |             nei_label = torch.sum(nei_label, dim=1)
 300 |             new_label = self.U1(h_atom) + self.U2(nei_label)
 301 |             h_atom = F.relu(new_label)
 302 |             h_atom = self.dropouts[i](h_atom)
 303 | 
 304 |         c_atom_final = layers[-1]
 305 |         if isinstance(atom_scope[0], list):
 306 |             c_mol = [torch.stack([c_atom_final[st: st + le].sum(dim=0)
 307 |                      for (st, le) in scope]) for scope in atom_scope]
 308 |             assert len(c_mol) == bs
 309 |             assert c_mol[0].shape[-1] == self.hsize
 310 |         else:
 311 |             c_mol = torch.stack([c_atom_final[st: st + le].sum(dim=0)
 312 |                                  for st, le in atom_scope])
 313 |             assert len(c_mol) == bs
 314 |             assert c_mol.shape[-1] == self.hsize
 315 | 
 316 |         if return_layers:
 317 |             return c_mol, layers
 318 | 
 319 |         return c_mol, layers[-1]
 320 | 
```

In [8]:
from seq_graph_retro.layers import WLNEncoder

# 实例化 WLN 编码器
encoder = WLNEncoder(
    n_atom_feat=ATOM_FDIM,  # 87
    n_bond_feat=BOND_FDIM,  # 6
    hsize=64,               # 隐藏层维度
    depth=3,                # 消息传递轮数
    bias=False,
    dropout_p=0.0,          # 教学演示关闭 dropout
)
encoder.eval()

# 前向传播
with torch.no_grad():
    c_mol, c_atom_layers = encoder(graph_tensors_undir, scopes_undir)

print("=== WLNEncoder 输出 ===")
display(pd.DataFrame([
    {"张量": "c_mol (分子级嵌入)", "形状": tuple(c_mol.shape), "说明": "对原子嵌入求和得到分子向量"},
    {"张量": "c_atom (原子级嵌入)", "形状": tuple(c_atom_layers.shape), "说明": "每个原子经过 depth 轮消息传递后的向量"},
]))

print(f"\n分子嵌入前 5 维: {c_mol[0, :5].numpy().round(3)}")
print(f"原子 0 嵌入前 5 维: {c_atom_layers[1, :5].numpy().round(3)}")


=== WLNEncoder 输出 ===


,张量,形状,说明
0,c_mol (分子级嵌入),"(1, 64)",对原子嵌入求和得到分子向量
1,c_atom (原子级嵌入),"(8, 64)",每个原子经过 depth 轮消息传递后的向量



分子嵌入前 5 维: [-0.     0.017  0.     0.01   0.004]
原子 0 嵌入前 5 维: [ 0.     0.002 -0.     0.002  0.   ]


In [9]:
# === 适配补丁 ===
# SingleEdit / LGIndEmbed 内部调用 WLNEncoder(node_fdim=..., edge_fdim=...)
# 但 WLNEncoder.__init__ 的实际参数名是 n_atom_feat / n_bond_feat
# 这里用 monkey-patch 做参数名映射，避免修改源代码

_original_wln_init = WLNEncoder.__init__

def _patched_wln_init(self, *args, **kwargs):
    if "node_fdim" in kwargs:
        kwargs["n_atom_feat"] = kwargs.pop("node_fdim")
    if "edge_fdim" in kwargs:
        kwargs["n_bond_feat"] = kwargs.pop("edge_fdim")
    return _original_wln_init(self, *args, **kwargs)

WLNEncoder.__init__ = _patched_wln_init
print("✅ WLNEncoder 参数名适配补丁已应用")

✅ WLNEncoder 参数名适配补丁已应用


WLNEncoder 的核心计算可以概括为：

```
输入: 原子特征 fnode [N, 87], 键特征 [N, 6]
     ↓ Linear embedding
h_atom [N, 64], h_bond [E, 64]
     ↓ 重复 depth=3 轮:
         gather 邻居原子和键的嵌入
         门控更新: c = W0(h) * Σ(W1(h_bond_nei) * W2(h_atom_nei))
     ↓ 最终原子嵌入 c_atom [N, 64]
     ↓ 按 scope 对原子求和
c_mol [batch, 64]  (分子级嵌入)
```

关键特点：**编码器只负责编码，不负责预测。** 预测由下游的 scoring network 完成。


## 5. Stage 1：键编辑预测 (SingleEdit)

Stage 1 是 GraphRetro 的核心：给定产物分子，预测应该对哪条键做什么操作。

**预测空间：** 对于一个有 $B$ 条键、$N$ 个原子的分子：
- 每条键有 5 种可能：变为 [无键(0.0), 单键(1.0), 双键(2.0), 三键(3.0), 芳香键(1.5)]
- 每个原子有 1 种可能：氢原子变化
- 总的 logits 维度 = $B \times 5 + N \times 1$

**模型架构：**
```
产物分子 → 图编码器 → 原子嵌入 c_atom [N, 64]
                              ↓
       ┌─────────────────────┬──────────────────┐
       │ 键打分: bond_score  │  原子打分:       │
       │ [sum(端点), diff(端点)] │  unimol_score   │
       │ → [B, 5]           │  → [N, 1]        │
       └─────────────────────┴──────────────────┘
                              ↓
                      拼接 → edit_logits [B×5 + N]
```


In [10]:
show_source_block(
    "source_repos/graphretro/seq_graph_retro/models/core_edits/single_edit.py",
    137, 222,
    "SingleEdit._compute_edit_logits + forward — Stage 1 核心"
)

### SingleEdit._compute_edit_logits + forward — Stage 1 核心
`source_repos/graphretro/seq_graph_retro/models/core_edits/single_edit.py:137-222`

```python
 137 |     def _compute_edit_logits(self, graph_tensors: Tuple[torch.Tensor],
 138 |                              scopes: Tuple[List],  bg_inputs: torch.Tensor = None,
 139 |                              ha: torch.Tensor = None) -> Tuple[torch.Tensor]:
 140 |         """
 141 |         Computes the edit logits.
 142 | 
 143 |         Parameters
 144 |         ----------
 145 |         graph_tensors: Tuple[torch.Tensor],
 146 |             Tensors representing a batch of graphs. Includes atom and bond features,
 147 |             and bond and atom neighbors
 148 |         scopes: Tuple[List],
 149 |             Scopes is composed of atom and bond scopes, which keep track of atom
 150 |             and bond indices for each molecule in the 2D feature list
 151 |         ha: torch.Tensor, default None
 152 |             Hidden states of atoms in the molecule
 153 |         """
 154 |         atom_scope, bond_scope = scopes
 155 | 
 156 |         c_mol, c_atom = self.encoder(graph_tensors, scopes)
 157 |         if self.toggles.get('use_attn', False):
 158 |             c_mol, c_atom_att = self.attn_layer(c_atom, scopes)
 159 |             c_atom_starts = index_select_ND(c_atom_att, dim=0, index=graph_tensors[-1][:, 0])
 160 |             c_atom_ends = index_select_ND(c_atom_att, dim=0, index=graph_tensors[-1][:, 1])
 161 | 
 162 |         else:
 163 |             c_atom_starts = index_select_ND(c_atom, dim=0, index=graph_tensors[-1][:, 0])
 164 |             c_atom_ends = index_select_ND(c_atom, dim=0, index=graph_tensors[-1][:, 1])
 165 | 
 166 |         sum_bonds = c_atom_starts + c_atom_ends
 167 |         diff_bonds = torch.abs(c_atom_starts - c_atom_ends)
 168 |         bond_score_inputs = torch.cat([sum_bonds, diff_bonds], dim=1)
 169 |         atom_score_inputs = c_atom.clone()
 170 | 
 171 |         if self.toggles.get("use_prod", False):
 172 |             atom_scope, bond_scope = scopes
 173 |             mol_exp_atoms = torch.cat([c_mol[idx].expand(le, -1)
 174 |                                     for idx, (st, le) in enumerate(atom_scope)], dim=0)
 175 |             mol_exp_bonds = torch.cat([c_mol[idx].expand(le, -1)
 176 |                                     for idx, (st, le) in enumerate(bond_scope)], dim=0)
 177 |             mol_exp_atoms = torch.cat([c_mol.new_zeros(1, c_mol.shape[-1]), mol_exp_atoms], dim=0)
 178 |             mol_exp_bonds = torch.cat([c_mol.new_zeros(1, c_mol.shape[-1]), mol_exp_bonds], dim=0)
 179 |             assert len(mol_exp_atoms) == len(atom_score_inputs)
 180 |             assert len(mol_exp_bonds) == len(bond_score_inputs)
 181 | 
 182 |             bond_score_inputs = torch.cat([bond_score_inputs, mol_exp_bonds], dim=-1)
 183 |             atom_score_inputs = torch.cat([atom_score_inputs, mol_exp_atoms], dim=-1)
 184 | 
 185 |         bond_logits = self.bond_score(bond_score_inputs)
 186 |         unimol_logits = self.unimol_score(atom_score_inputs)
 187 | 
 188 |         if self.toggles.get("propagate_logits", False):
 189 |             bg_tensors, bg_scope = bg_inputs
 190 |             assert len(bond_logits) == len(bg_tensors[0])
 191 |             bond_logits = self.bond_label_mpn(bond_logits, bg_tensors, mask=None)
 192 |             edit_logits = [torch.cat([bond_logits[st_b: st_b+le_b].flatten(),
 193 |                                      unimol_logits[st_a: st_a+le_a].flatten()], dim=-1)
 194 |                            for ((st_a, le_a), (st_b, le_b)) in zip(*(atom_scope, bond_scope))]
 195 |             return c_mol, edit_logits, None
 196 | 
 197 |         edit_logits = [torch.cat([bond_logits[st_b: st_b+le_b].flatten(),
 198 |                                  unimol_logits[st_a: st_a+le_a].flatten()], dim=-1)
 199 |                        for ((st_a, le_a), (st_b, le_b)) in zip(*(atom_scope, bond_scope))]
 200 | 
 201 |         return c_mol, edit_logits, None
 202 | 
 203 |     def forward(self, graph_tensors: Tuple[torch.Tensor], scopes: Tuple[List],
 204 |                 bg_inputs = None) -> Tuple[torch.Tensor]:
 205 |         """Forward pass
 206 | 
 207 |         Parameters
 208 |         ----------
 209 |         graph_tensors: Tuple[torch.Tensor],
 210 |             Tensors representing a batch of graphs. Includes atom and bond features,
 211 |             and bond and atom neighbors
 212 |         scopes: Tuple[List],
 213 |             Scopes is composed of atom and bond scopes, which keep track of atom
 214 |             and bond indices for each molecule in the 2D feature list
 215 |         """
 216 |         graph_tensors = self.to_device(graph_tensors)
 217 |         if bg_inputs is not None:
 218 |             bg_tensors, bg_scope = bg_inputs
 219 |             bg_tensors = self.to_device(bg_tensors)
 220 |             bg_inputs = (bg_tensors, bg_scope)
 221 |         c_mol, edit_logits, _ = self._compute_edit_logits(graph_tensors, scopes,
 222 |                                                        ha=None, bg_inputs=bg_inputs)
```

In [11]:
from seq_graph_retro.models import SingleEdit

# Stage 1 的默认配置
config_stage1 = {
    "n_atom_feat": ATOM_FDIM,
    "n_bond_feat": BOND_FDIM,
    "rnn_type": "gru",
    "mpn_size": 64,
    "depth": 3,
    "dropout_mpn": 0.0,
    "mlp_size": 128,
    "dropout_mlp": 0.0,
    "bs_outdim": 5,
    "edit_loss": "sigmoid",
    "pos_weight": 5,
    "bias": False,
    "n_heads": 8,
    "n_mt_blocks": 3,
}
toggles_stage1 = {
    "use_attn": False,
    "use_prod": False,
    "use_concat": False,
    "propagate_logits": False,
}

model_stage1 = SingleEdit(
    config=config_stage1,
    encoder_name="WLNEncoder",
    toggles=toggles_stage1,
    device="cpu",
)
model_stage1.eval()

# 前向传播
with torch.no_grad():
    prod_vecs, edit_logits = model_stage1(graph_tensors_undir, scopes_undir)

print("=== Stage 1 输出 ===")
num_bonds = prod_mol.GetNumBonds()
num_atoms = prod_mol.GetNumAtoms()

display(pd.DataFrame([
    {"张量": "prod_vecs (分子嵌入)", "形状": tuple(prod_vecs.shape)},
    {"张量": "edit_logits", "形状": tuple(edit_logits[0].shape),
     "说明": f"num_bonds×5 + num_atoms = {num_bonds}×5 + {num_atoms} = {num_bonds*5 + num_atoms}"},
]))

# 解码最高分的编辑
logits_np = edit_logits[0].numpy()
top_idx = int(logits_np.argmax())
if top_idx < num_bonds * 5:
    bond_idx = top_idx // 5
    bo_idx = top_idx % 5
    bond = prod_mol.GetBondWithIdx(bond_idx)
    a1 = bond.GetBeginAtom().GetAtomMapNum()
    a2 = bond.GetEndAtom().GetAtomMapNum()
    new_bo = BOND_FLOATS[bo_idx]
    old_bo = bond.GetBondTypeAsDouble()
    print(f"\n预测最高分编辑 (随机权重): 键 {bond_idx} (原子{a1}-原子{a2})")
    print(f"  当前键级: {old_bo} → 预测键级: {new_bo}")
    print(f"  编辑字符串: \"{a1}:{a2}:{old_bo}:{new_bo}\"")
else:
    atom_idx = top_idx - num_bonds * 5
    amap = prod_mol.GetAtomWithIdx(atom_idx).GetAtomMapNum()
    print(f"\n预测最高分编辑 (随机权重): 原子 {atom_idx} (map={amap}) 的氢变化")

# 展示 top-5
top5_indices = logits_np.argsort()[-5:][::-1]
top5_info = []
for idx in top5_indices:
    idx = int(idx)
    if idx < num_bonds * 5:
        bi = idx // 5
        boi = idx % 5
        b = prod_mol.GetBondWithIdx(bi)
        info = f"键{bi}: 原子{b.GetBeginAtom().GetAtomMapNum()}-{b.GetEndAtom().GetAtomMapNum()} → {BOND_FLOATS[boi]}"
    else:
        ai = idx - num_bonds * 5
        info = f"原子{ai}: 氢变化"
    top5_info.append({"rank": len(top5_info)+1, "logit_idx": idx, "score": round(float(logits_np[idx]), 4), "编辑": info})
display(pd.DataFrame(top5_info))

=== Stage 1 输出 ===


,张量,形状,说明
0,prod_vecs (分子嵌入),"(1, 64)",NaN
1,edit_logits,"(42,)",num_bonds×5 + num_atoms = 7×5 + 7 = 42



预测最高分编辑 (随机权重): 键 3 (原子4-原子5)
  当前键级: 1.0 → 预测键级: 0.0
  编辑字符串: "4:5:1.0:0.0"


,rank,logit_idx,score,编辑
0,1,15,0.0672,键3: 原子4-5 → 0.0
1,2,25,0.0672,键5: 原子6-7 → 0.0
2,3,30,0.0671,键6: 原子7-1 → 0.0
3,4,10,0.0671,键2: 原子3-4 → 0.0
4,5,0,0.0671,键0: 原子1-2 → 0.0


Stage 1 的预测流程可以概括为 4 步：

1. **编码** — 用 WLNEncoder 把产物编码为原子嵌入 `c_atom [N, 64]`
2. **键打分** — 对每条键的两端原子嵌入做 `sum + diff`，通过 MLP 打 5 个分
3. **原子打分** — 对每个原子嵌入通过另一个 MLP 打 1 个分（氢变化）
4. **拼接** — 把所有分数拼成一个大向量，argmax 找到最佳编辑

**重要区分：** SingleEdit 预测的是 **"对产物做什么编辑"**，而不是直接预测反应物。编辑被应用后，产物才变成 synthons。


## 6. 数据基础：反应解析与编辑提取

在理解模型如何预测编辑之前，我们需要知道**训练标签是怎么来的**。

`get_reaction_core()` 通过比较反应物和产物的键信息，自动提取出发生变化的键编辑：


In [12]:
show_source_block(
    "source_repos/graphretro/seq_graph_retro/utils/parse.py",
    74, 156,
    "get_reaction_core — 提取键编辑标签"
)

### get_reaction_core — 提取键编辑标签
`source_repos/graphretro/seq_graph_retro/utils/parse.py:74-156`

```python
  74 | def get_reaction_core(r: str, p: str, kekulize: bool = False, use_h_labels: bool = False) -> Tuple[Set, List]:
  75 |     """Get the reaction core and edits for given reaction
  76 | 
  77 |     Parameters
  78 |     ----------
  79 |     r: str,
  80 |         SMILES string representing the reactants
  81 |     p: str,
  82 |         SMILES string representing the product
  83 |     kekulize: bool,
  84 |         Whether to kekulize molecules to fetch minimal set of edits
  85 |     use_h_labels: bool,
  86 |         Whether to use change in hydrogen counts in edits
  87 |     """
  88 |     reac_mol = get_mol(r)
  89 |     prod_mol = get_mol(p)
  90 | 
  91 |     if reac_mol is None or prod_mol is None:
  92 |         return set(), []
  93 | 
  94 |     if kekulize:
  95 |         reac_mol, prod_mol = align_kekule_pairs(r, p)
  96 | 
  97 |     prod_bonds = get_bond_info(prod_mol)
  98 |     p_amap_idx = {atom.GetAtomMapNum(): atom.GetIdx() for atom in prod_mol.GetAtoms()}
  99 | 
 100 |     max_amap = max([atom.GetAtomMapNum() for atom in reac_mol.GetAtoms()])
 101 |     for atom in reac_mol.GetAtoms():
 102 |         if atom.GetAtomMapNum() == 0:
 103 |             atom.SetAtomMapNum(max_amap + 1)
 104 |             max_amap += 1
 105 | 
 106 |     reac_bonds = get_bond_info(reac_mol)
 107 |     reac_amap = {atom.GetAtomMapNum(): atom.GetIdx() for atom in reac_mol.GetAtoms()}
 108 | 
 109 |     rxn_core = set()
 110 |     core_edits = []
 111 | 
 112 |     for bond in prod_bonds:
 113 |         if bond in reac_bonds and prod_bonds[bond][0] != reac_bonds[bond][0]:
 114 |             a_start, a_end = bond
 115 |             prod_bo, reac_bo = prod_bonds[bond][0], reac_bonds[bond][0]
 116 | 
 117 |             a_start, a_end = sorted([a_start, a_end])
 118 |             edit = f"{a_start}:{a_end}:{prod_bo}:{reac_bo}"
 119 |             core_edits.append(edit)
 120 |             rxn_core.update([a_start, a_end])
 121 | 
 122 |         if bond not in reac_bonds:
 123 |             a_start, a_end = bond
 124 |             reac_bo = 0.0
 125 |             prod_bo = prod_bonds[bond][0]
 126 | 
 127 |             start, end = sorted([a_start, a_end])
 128 |             edit = f"{a_start}:{a_end}:{prod_bo}:{reac_bo}"
 129 |             core_edits.append(edit)
 130 |             rxn_core.update([a_start, a_end])
 131 | 
 132 |     for bond in reac_bonds:
 133 |         if bond not in prod_bonds:
 134 |             amap1, amap2 = bond
 135 | 
 136 |             if (amap1 in p_amap_idx) and (amap2 in p_amap_idx):
 137 |                 a_start, a_end = sorted([amap1, amap2])
 138 |                 reac_bo = reac_bonds[bond][0]
 139 |                 edit = f"{a_start}:{a_end}:{0.0}:{reac_bo}"
 140 |                 core_edits.append(edit)
 141 |                 rxn_core.update([a_start, a_end])
 142 | 
 143 |     if use_h_labels:
 144 |         if len(rxn_core) == 0:
 145 |             for atom in prod_mol.GetAtoms():
 146 |                 amap_num = atom.GetAtomMapNum()
 147 | 
 148 |                 numHs_prod = atom.GetTotalNumHs()
 149 |                 numHs_reac = reac_mol.GetAtomWithIdx(reac_amap[amap_num]).GetTotalNumHs()
 150 | 
 151 |                 if numHs_prod != numHs_reac:
 152 |                     edit = f"{amap_num}:{0}:{1.0}:{0.0}"
 153 |                     core_edits.append(edit)
 154 |                     rxn_core.add(amap_num)
 155 | 
 156 |     return rxn_core, core_edits
```

In [13]:
from seq_graph_retro.utils.parse import get_reaction_core, get_reaction_info

# 用 demo_data 中的第一条反应
demo_df = pd.read_csv(DATA_DIR / "demo_data.csv")
rxn = demo_df.iloc[0]["reactants>reagents>production"]
parts = rxn.split(">>")
reactants_smi = parts[0]
product_smi = parts[-1]

print(f"反应: {demo_df.iloc[0]['id']}")
print(f"反应物: {reactants_smi[:80]}...")
print(f"产物:   {product_smi[:80]}...")
print()

# 提取反应核心和编辑
rxn_core, core_edits = get_reaction_core(reactants_smi, product_smi)
print(f"反应核心原子 (atom map numbers): {rxn_core}")
print(f"键编辑数: {len(core_edits)}")
for i, edit in enumerate(core_edits):
    a1, a2, prev_bo, new_bo = edit.split(":")
    print(f"  编辑 {i}: 原子{a1}-原子{a2}, 键级 {prev_bo} → {new_bo}")

# 完整的反应信息
full_rxn_smi = f"{reactants_smi}>>{product_smi}"
rxn_info = get_reaction_info(full_rxn_smi)
print(f"\n完整反应信息:")
print(f"  core_edits: {rxn_info.core_edits}")
print(f"  attach_atoms: {rxn_info.attach_atoms}")


反应: US07928231B2
反应物: [C:12](=[O:13])([O:14][C:15]([CH3:16])([CH3:17])[CH3:18])[O:27][C:25]([O:24][C:2...
产物:   [CH3:1][C:2](=[O:3])[c:4]1[cH:5][cH:6][c:7]2[c:8]([cH:9][cH:10][n:11]2[C:12](=[O...

反应核心原子 (atom map numbers): {11, 12}
键编辑数: 1
  编辑 0: 原子11-原子12, 键级 1.0 → 0.0

完整反应信息:
  core_edits: ['11:12:1.0:0.0']
  attach_atoms: [[], [11]]


## 7. 化学工具：apply_edits_to_mol

当 Stage 1 预测出键编辑后，需要把编辑**实际应用到分子上**，得到中间碎片 (synthons)。

这一步并不简单——需要处理芳香氮、共轭体系、超价等特殊化学情况。

`apply_edits_to_mol()` 是连接 Stage 1 和 Stage 2 的桥梁函数：


In [14]:
show_source_block(
    "source_repos/graphretro/seq_graph_retro/utils/chem.py",
    30, 110,
    "apply_edits_to_mol — 将编辑应用到分子"
)

### apply_edits_to_mol — 将编辑应用到分子
`source_repos/graphretro/seq_graph_retro/utils/chem.py:30-110`

```python
  30 | def apply_edits_to_mol(mol: Chem.Mol, edits: Iterable[str]) -> Chem.Mol:
  31 |     """Apply edits to molecular graph.
  32 | 
  33 |     Parameters
  34 |     ----------
  35 |     mol: Chem.Mol,
  36 |         RDKit mol object
  37 |     edits: Iterable[str],
  38 |         Iterable of edits to apply. An edit is structured as a1:a2:b1:b2, where
  39 |         a1, a2 are atom maps of participating atoms and b1, b2 are previous and
  40 |         new bond orders. When  a2 = 0, we update the hydrogen count.
  41 |     """
  42 |     new_mol = Chem.RWMol(mol)
  43 |     amap = {atom.GetAtomMapNum(): atom.GetIdx() for atom in new_mol.GetAtoms()}
  44 | 
  45 |     # Keep track of aromatic nitrogens, might cause explicit hydrogen issues
  46 |     aromatic_nitrogen_idx = set()
  47 |     aromatic_carbonyl_adj_to_aromatic_nH = {}
  48 |     aromatic_carbondeg3_adj_to_aromatic_nH0 = {}
  49 |     for a in new_mol.GetAtoms():
  50 |         if a.GetIsAromatic() and a.GetSymbol() == 'N':
  51 |             aromatic_nitrogen_idx.add(a.GetIdx())
  52 |             for nbr in a.GetNeighbors():
  53 |                 nbr_is_carbon = (nbr.GetSymbol() == 'C')
  54 |                 nbr_is_aromatic = nbr.GetIsAromatic()
  55 |                 nbr_has_double_bond = any(b.GetBondTypeAsDouble() == 2 for b in nbr.GetBonds())
  56 |                 nbr_has_3_bonds = (len(nbr.GetBonds()) == 3)
  57 | 
  58 |                 if (a.GetNumExplicitHs() ==1 and nbr_is_carbon and nbr_is_aromatic
  59 |                     and nbr_has_double_bond):
  60 |                     aromatic_carbonyl_adj_to_aromatic_nH[nbr.GetIdx()] = a.GetIdx()
  61 |                 elif (a.GetNumExplicitHs() == 0 and nbr_is_carbon and nbr_is_aromatic
  62 |                     and nbr_has_3_bonds):
  63 |                     aromatic_carbondeg3_adj_to_aromatic_nH0[nbr.GetIdx()] = a.GetIdx()
  64 |         else:
  65 |             a.SetNumExplicitHs(0)
  66 |     new_mol.UpdatePropertyCache()
  67 | 
  68 |     # Apply the edits as predicted
  69 |     for edit in edits:
  70 |         x, y, prev_bo, new_bo = edit.split(":")
  71 |         x, y = int(x), int(y)
  72 |         new_bo = float(new_bo)
  73 | 
  74 |         if y == 0:
  75 |             continue
  76 | 
  77 |         bond = new_mol.GetBondBetweenAtoms(amap[x],amap[y])
  78 |         a1 = new_mol.GetAtomWithIdx(amap[x])
  79 |         a2 = new_mol.GetAtomWithIdx(amap[y])
  80 | 
  81 |         if bond is not None:
  82 |             new_mol.RemoveBond(amap[x],amap[y])
  83 | 
  84 |             # Are we losing a bond on an aromatic nitrogen?
  85 |             if bond.GetBondTypeAsDouble() == 1.0:
  86 |                 if amap[x] in aromatic_nitrogen_idx:
  87 |                     if a1.GetTotalNumHs() == 0:
  88 |                         a1.SetNumExplicitHs(1)
  89 |                     elif a1.GetFormalCharge() == 1:
  90 |                         a1.SetFormalCharge(0)
  91 |                 elif amap[y] in aromatic_nitrogen_idx:
  92 |                     if a2.GetTotalNumHs() == 0:
  93 |                         a2.SetNumExplicitHs(1)
  94 |                     elif a2.GetFormalCharge() == 1:
  95 |                         a2.SetFormalCharge(0)
  96 | 
  97 |             # Are we losing a c=O bond on an aromatic ring? If so, remove H from adjacent nH if appropriate
  98 |             if bond.GetBondTypeAsDouble() == 2.0:
  99 |                 if amap[x] in aromatic_carbonyl_adj_to_aromatic_nH:
 100 |                     new_mol.GetAtomWithIdx(aromatic_carbonyl_adj_to_aromatic_nH[amap[x]]).SetNumExplicitHs(0)
 101 |                 elif amap[y] in aromatic_carbonyl_adj_to_aromatic_nH:
 102 |                     new_mol.GetAtomWithIdx(aromatic_carbonyl_adj_to_aromatic_nH[amap[y]]).SetNumExplicitHs(0)
 103 | 
 104 |         if new_bo > 0:
 105 |             new_mol.AddBond(amap[x],amap[y],BOND_FLOAT_TO_TYPE[new_bo])
 106 | 
 107 |             # Special alkylation case?
 108 |             if new_bo == 1:
 109 |                 if amap[x] in aromatic_nitrogen_idx:
 110 |                     if a1.GetTotalNumHs() == 1:
```

In [15]:
from seq_graph_retro.utils.chem import apply_edits_to_mol, get_mol

# 用一个简单的例子演示编辑应用
simple_prod = get_mol(product_smi)
print(f"产物 SMILES: {product_smi[:80]}...")
print(f"产物原子数: {simple_prod.GetNumAtoms()}")
print(f"将要应用的编辑: {core_edits}")
print()

try:
    fragments_mol = apply_edits_to_mol(simple_prod, core_edits)
    frag_smi = Chem.MolToSmiles(fragments_mol)
    print(f"编辑后碎片 SMILES: {frag_smi}")
    
    # 查看碎片（连通分量）
    frags = Chem.GetMolFrags(fragments_mol, asMols=True, sanitizeFrags=False)
    print(f"碎片数量: {len(frags)}")
    for i, frag in enumerate(frags):
        print(f"  碎片 {i}: {Chem.MolToSmiles(frag)}")
except Exception as e:
    print(f"编辑应用出错 (这在教学演示中是正常的): {e}")
    print("在真实推理中，模型预测的编辑通常是化学上合理的")


产物 SMILES: [CH3:1][C:2](=[O:3])[c:4]1[cH:5][cH:6][c:7]2[c:8]([cH:9][cH:10][n:11]2[C:12](=[O...
产物原子数: 19
将要应用的编辑: ['11:12:1.0:0.0']

编辑后碎片 SMILES: [C:12](=[O:13])[O:14][C:15]([C:16])([C:17])[C:18].[C:1][C:2](=[O:3])[c:4]1[c:5][c:6][c:7]2[c:8]([c:9][c:10][nH:11]2)[c:19]1
碎片数量: 2
  碎片 0: [C:1][C:2](=[O:3])[c:4]1[c:5][c:6][c:7]2[c:8]([c:9][c:10][nH:11]2)[c:19]1
  碎片 1: [C:12](=[O:13])[O:14][C:15]([C:16])([C:17])[C:18]


## 8. Stage 2：离去基团预测 (LGIndEmbed)

Stage 2 接收 Stage 1 产生的 synthons（碎片），为每个碎片预测需要补上的 **leaving group（离去基团）**。

**关键设计：自回归生成**

如果反应产生了多个碎片，LGIndEmbed 会**逐个**预测它们的离去基团，并且每一步的预测会参考前一步的结果。

```
初始化: prev_lg_emb = embed(<bos>)

对每个碎片 i:
  context = [prev_lg_emb, prod_vecs, frag_vecs[i]]
  scores = lg_score(context)  → [vocab_size]
  
  训练时: 用 gold label (Teacher Forcing)
  推理时: 用 argmax(scores)
  
  prev_lg_emb = embed(选中的LG)
```

这就像一个**序列生成模型**，只不过生成的是离去基团的索引，而不是词语。


In [16]:
show_source_block(
    "source_repos/graphretro/seq_graph_retro/models/lg_edits/lg_ind_embed.py",
    16, 130,
    "LGIndEmbed — Stage 2 离去基团预测"
)

### LGIndEmbed — Stage 2 离去基团预测
`source_repos/graphretro/seq_graph_retro/models/lg_edits/lg_ind_embed.py:16-130`

```python
  16 | class LGIndEmbed(nn.Module):
  17 |     """LGIndEmbed is a classifier for predicting leaving groups on fragments."""
  18 | 
  19 |     def __init__(self,
  20 |                  config: Dict,
  21 |                  lg_vocab: Vocab,
  22 |                  encoder_name: str,
  23 |                  toggles: Dict = None,
  24 |                  device: str = 'cpu',
  25 |                  **kwargs):
  26 |         """
  27 |         Parameters
  28 |         ----------
  29 |         config: Dict,
  30 |             Config for all sub-modules and self
  31 |         lg_vocab: Vocab
  32 |             Vocabulary of leaving groups
  33 |         encoder_name: str,
  34 |             Name of the encoder network
  35 |         use_prev_pred: bool, default True
  36 |             Whether to use previous leaving group prediction
  37 |         device: str
  38 |             Device on which program runs
  39 |         """
  40 |         super(LGIndEmbed, self).__init__(**kwargs)
  41 |         self.config = config
  42 |         self.lg_vocab = lg_vocab
  43 |         self.encoder_name = encoder_name
  44 |         self.toggles = toggles if toggles is not None else {}
  45 |         self.device = device
  46 |         self.E_lg = torch.eye(len(lg_vocab)).to(device)
  47 | 
  48 |         self._build_layers()
  49 | 
  50 |     def _build_layers(self) -> None:
  51 |         """Builds the layers in the classifier."""
  52 |         config = self.config
  53 |         if self.encoder_name == 'GraphFeatEncoder':
  54 |             self.encoder = GraphFeatEncoder(node_fdim=config['n_atom_feat'],
  55 |                                             edge_fdim=config['n_bond_feat'],
  56 |                                             rnn_type=config['rnn_type'],
  57 |                                             hsize=config['mpn_size'],
  58 |                                             depth=config['depth'],
  59 |                                             dropout_p=config['dropout_mpn'])
  60 | 
  61 |         elif self.encoder_name == 'WLNEncoder':
  62 |             self.encoder = WLNEncoder(node_fdim=config['n_atom_feat'],
  63 |                                       edge_fdim=config['n_bond_feat'],
  64 |                                       hsize=config['mpn_size'],
  65 |                                       depth=config['depth'],
  66 |                                       bias=config['bias'],
  67 |                                       dropout_p=config['dropout_mpn'])
  68 |         else:
  69 |             raise ValueError()
  70 | 
  71 |         if self.toggles.get('use_attn', False):
  72 |             self.attn_layer = AtomAttention(n_bin_feat=config['n_bin_feat'],
  73 |                                             hsize=config['mpn_size'],
  74 |                                             n_heads=config['n_heads'],
  75 |                                             bias=config['bias'])
  76 | 
  77 |         lg_score_in_dim = 2 * config['mpn_size']
  78 |         if self.toggles.get('use_prev_pred', False):
  79 |             lg_score_in_dim += config['embed_size']
  80 | 
  81 |         self.lg_embedding = nn.Linear(in_features=len(self.lg_vocab),
  82 |                                       out_features=config['embed_size'],
  83 |                                       bias=config['embed_bias'])
  84 | 
  85 |         self.lg_score = build_mlp(in_dim=lg_score_in_dim,
  86 |                                   h_dim=config['mlp_size'],
  87 |                                   out_dim=len(self.lg_vocab),
  88 |                                   dropout_p=config['dropout_mlp'])
  89 | 
  90 |         self.lg_loss = nn.CrossEntropyLoss(ignore_index=self.lg_vocab["<pad>"])
  91 | 
  92 |     def _compute_lg_step(self, graph_vecs, prod_vecs, prev_embed=None):
  93 |         if self.toggles.get('use_prev_pred', False):
  94 |             if prev_embed is None:
  95 |                 init_state = torch.zeros(graph_vecs.size(0), len(self.lg_vocab), device=self.device)
  96 |                 init_state[:, self.lg_vocab.get("<bos>")] = 1
  97 |                 prev_lg_emb = self.lg_embedding(init_state)
  98 |             else:
  99 |                 prev_lg_emb = prev_embed
 100 | 
 101 |         if self.toggles.get('use_prev_pred', False):
 102 |             scores_lg = self.lg_score(torch.cat([prev_lg_emb, prod_vecs, graph_vecs], dim=-1))
 103 |         else:
 104 |             scores_lg = self.lg_score(torch.cat([prod_vecs, graph_vecs], dim=-1))
 105 |         return scores_lg, None
 106 | 
 107 |     def _compute_lg_logits(self, graph_vecs_pad, prod_vecs, lg_labels=None):
 108 |         scores = torch.tensor([], device=self.device)
 109 |         prev_lg_emb = None
 110 | 
 111 |         if lg_labels is None:
 112 |             for idx in range(graph_vecs_pad.size(1)):
 113 |                 scores_lg, _ = self._compute_lg_step(graph_vecs_pad[:, idx], prod_vecs, prev_embed=prev_lg_emb)
 114 |                 prev_lg_emb = self.lg_embedding(self.E_lg.index_select(index=torch.argmax(scores_lg, dim=-1), dim=0))
 115 |                 scores = torch.cat([scores, scores_lg.unsqueeze(1)], dim=1)
 116 | 
 117 |         else:
 118 |             for idx in range(graph_vecs_pad.size(1)):
 119 |                 scores_lg, _ = self._compute_lg_step(graph_vecs_pad[:, idx], prod_vecs, prev_embed=prev_lg_emb)
 120 |                 prev_lg_emb = self.lg_embedding(self.E_lg.index_select(index=lg_labels[:, idx], dim=0))
 121 |                 scores = torch.cat([scores, scores_lg.unsqueeze(1)], dim=1)
 122 | 
 123 |         return scores
 124 | 
 125 |     def forward(self, prod_inputs, frag_inputs):
 126 |         prod_tensors, prod_scopes = prod_inputs
 127 |         frag_tensors, frag_scopes = frag_inputs
 128 | 
 129 |         prod_tensors = self.to_device(prod_tensors)
 130 |         frag_tensors = self.to_device(frag_tensors)
```

In [17]:
from seq_graph_retro.molgraph.vocab import Vocab

# 构建一个小型离去基团词汇表（真实场景从训练数据统计）
# 注意: "<pad>" 是 LGIndEmbed 内部 lg_loss 需要的特殊 token
demo_lg_list = [
    "<pad>",
    "[H]", "O", "Cl", "Br", "I", "N", "C", "CC",
    "OC(C)=O", "[NH2]", "F", "S", "OC=O",
    "c1ccccc1", "[OH]", "OC(=O)C(F)(F)F",
]
lg_vocab = Vocab(demo_lg_list)
print(f"离去基团词汇表大小: {len(lg_vocab)}")
print(f"词汇表示例:")
display(pd.DataFrame([
    {"index": i, "LG_SMILES": lg_vocab.get_elem(i)} 
    for i in range(min(10, len(lg_vocab)))
]))

from seq_graph_retro.models import LGIndEmbed

config_stage2 = {
    "n_atom_feat": ATOM_FDIM,
    "n_bond_feat": BOND_FDIM,
    "rnn_type": "gru",
    "mpn_size": 64,
    "depth": 3,
    "dropout_mpn": 0.0,
    "embed_size": 32,
    "mlp_size": 128,
    "dropout_mlp": 0.0,
    "bias": False,
    "embed_bias": False,
    "n_heads": 8,
    "n_mt_blocks": 3,
}
toggles_stage2 = {"use_prev_pred": True}

model_stage2 = LGIndEmbed(
    config=config_stage2,
    lg_vocab=lg_vocab,
    encoder_name="WLNEncoder",
    toggles=toggles_stage2,
    device="cpu",
)
model_stage2.eval()

print(f"\n=== LGIndEmbed 模型结构概览 ===")
print(f"  编码器: WLNEncoder (hsize=64, depth=3)")
print(f"  LG 嵌入维度: {config_stage2['embed_size']}")
print(f"  LG 词汇表大小: {len(lg_vocab)}")
print(f"  LG 打分网络输入: embed_size + 2×hsize = {config_stage2['embed_size'] + 2*config_stage2['mpn_size']}")
print(f"  LG 打分网络输出: {len(lg_vocab)} (每个 LG 一个分数)")

离去基团词汇表大小: 17
词汇表示例:


,index,LG_SMILES
0,0,<pad>
1,1,[H]
2,2,O
3,3,Cl
4,4,Br
5,5,I
6,6,N
7,7,C
8,8,CC
9,9,OC(C)=O



=== LGIndEmbed 模型结构概览 ===
  编码器: WLNEncoder (hsize=64, depth=3)
  LG 嵌入维度: 32
  LG 词汇表大小: 17
  LG 打分网络输入: embed_size + 2×hsize = 160
  LG 打分网络输出: 17 (每个 LG 一个分数)


Stage 2 的 `_compute_lg_logits` 方法实现了自回归循环：

```python
prev_embed = lg_embedding(E_lg[bos_idx])  # 初始嵌入

for step_i in range(max_fragments):
    # 拼接上下文: [前一步LG嵌入, 产物嵌入, 当前碎片嵌入]
    context = torch.cat([prev_embed, prod_vecs, frag_vecs[:, step_i]], dim=-1)
    scores = lg_score(context)  # → [batch, vocab_size]
    
    if training:
        pred_idx = gold_labels[:, step_i]    # Teacher Forcing
    else:
        pred_idx = scores.argmax(dim=-1)     # 贪心解码
    
    prev_embed = lg_embedding(E_lg[pred_idx])  # 更新嵌入
```

**Teacher Forcing 的好处：** 训练时用正确答案作为下一步输入，避免错误累积，训练更稳定。  
**推理时的变化：** 没有正确答案可用，只能用自己的预测结果，因此可能出现错误传播。


## 9. 顶层总装：完整推理流程

现在可以把 Stage 1 和 Stage 2 连起来，看看一次完整的逆合成推理是怎样工作的：

```
产物 SMILES
     ↓
[Stage 1: SingleEdit]
     │ 1. pack_graph_feats → 图张量
     │ 2. WLNEncoder → 原子嵌入 c_atom
     │ 3. bond_score + unimol_score → edit_logits
     │ 4. argmax → 最佳编辑 "a1:a2:prev_bo:new_bo"
     ↓
[apply_edits_to_mol] → Synthons (碎片)
     ↓
[Stage 2: LGIndEmbed]
     │ 1. 分别编码产物和碎片
     │ 2. 自回归预测每个碎片的 LG
     │ 3. 从词汇表中选择最佳 LG
     ↓
[组合] 碎片 + LG → 反应物
```


In [18]:
# 展示完整推理流程的关键张量形状
print("=" * 60)
print("GraphRetro 完整推理流程 — 张量流")
print("=" * 60)

pipeline_steps = [
    {"步骤": "1. 产物特征化",
     "输入": f"SMILES '{prod_smi}'",
     "输出": f"fnode [{prod_mol.GetNumAtoms()}, {ATOM_FDIM}]",
     "函数": "pack_graph_feats()"},
    {"步骤": "2. 图编码",
     "输入": f"graph_tensors (5 个张量)",
     "输出": f"c_atom [{prod_mol.GetNumAtoms()}, 64], c_mol [1, 64]",
     "函数": "WLNEncoder.forward()"},
    {"步骤": "3. 键编辑预测",
     "输入": f"c_atom [{prod_mol.GetNumAtoms()}, 64]",
     "输出": f"edit_logits [{num_bonds*5 + num_atoms}]",
     "函数": "SingleEdit._compute_edit_logits()"},
    {"步骤": "4. 应用编辑",
     "输入": "edit_string '1:2:1.0:0.0'",
     "输出": "fragments (RDKit Mol)",
     "函数": "apply_edits_to_mol()"},
    {"步骤": "5. 碎片编码",
     "输入": "碎片图张量",
     "输出": f"frag_vecs_pad [1, max_frags, 64]",
     "函数": "LGIndEmbed.forward()"},
    {"步骤": "6. LG 预测",
     "输入": f"[prev_lg_emb, prod_vecs, frag_vecs]",
     "输出": f"lg_scores [1, max_frags, {len(lg_vocab)}]",
     "函数": "LGIndEmbed._compute_lg_logits()"},
    {"步骤": "7. 组合输出",
     "输入": "碎片 + LG SMILES",
     "输出": "反应物 SMILES",
     "函数": "组合与后处理"},
]
display(pd.DataFrame(pipeline_steps))


GraphRetro 完整推理流程 — 张量流


,步骤,输入,输出,函数
0,1. 产物特征化,SMILES 'c1ccc(O)cc1',"fnode [7, 98]",pack_graph_feats()
1,2. 图编码,graph_tensors (5 个张量),"c_atom [7, 64], c_mol [1, 64]",WLNEncoder.forward()
2,3. 键编辑预测,"c_atom [7, 64]",edit_logits [42],SingleEdit._compute_edit_logits()
3,4. 应用编辑,edit_string '1:2:1.0:0.0',fragments (RDKit Mol),apply_edits_to_mol()
4,5. 碎片编码,碎片图张量,"frag_vecs_pad [1, max_frags, 64]",LGIndEmbed.forward()
5,6. LG 预测,"[prev_lg_emb, prod_vecs, frag_vecs]","lg_scores [1, max_frags, 17]",LGIndEmbed._compute_lg_logits()
6,7. 组合输出,碎片 + LG SMILES,反应物 SMILES,组合与后处理


In [19]:
# 完整的 pipeline 配置汇总
pipeline_summary = pd.DataFrame([
    {
        "step": "1. 原子/键特征提取",
        "source_symbol": "mol_features.get_atom_features / get_bond_features",
        "intermediate": f"fnode [N, {ATOM_FDIM}], fmess [E, {BOND_FDIM}]",
        "teaching_mapping": "show_source_block: mol_features.py",
    },
    {
        "step": "2. 图 batch 构建",
        "source_symbol": "collate_fns.pack_graph_feats",
        "intermediate": "graph_tensors (5个张量) + scopes",
        "teaching_mapping": "show_source_block: collate_fns.py",
    },
    {
        "step": "3. GNN 编码",
        "source_symbol": "WLNEncoder.forward",
        "intermediate": "c_mol [B, 64], c_atom [N, 64]",
        "teaching_mapping": "show_source_block: encoder.py",
    },
    {
        "step": "4. 键编辑打分",
        "source_symbol": "SingleEdit._compute_edit_logits",
        "intermediate": "edit_logits [B×5 + N]",
        "teaching_mapping": "show_source_block: single_edit.py",
    },
    {
        "step": "5. 编辑应用",
        "source_symbol": "chem.apply_edits_to_mol",
        "intermediate": "Synthon fragments",
        "teaching_mapping": "show_source_block: chem.py",
    },
    {
        "step": "6. LG 自回归预测",
        "source_symbol": "LGIndEmbed._compute_lg_logits",
        "intermediate": "lg_scores [B, max_frags, vocab]",
        "teaching_mapping": "show_source_block: lg_ind_embed.py",
    },
])
display(pipeline_summary)


,step,source_symbol,intermediate,teaching_mapping
0,1. 原子/键特征提取,mol_features.get_atom_features / get_bond_feat...,"fnode [N, 98], fmess [E, 6]",show_source_block: mol_features.py
1,2. 图 batch 构建,collate_fns.pack_graph_feats,graph_tensors (5个张量) + scopes,show_source_block: collate_fns.py
2,3. GNN 编码,WLNEncoder.forward,"c_mol [B, 64], c_atom [N, 64]",show_source_block: encoder.py
3,4. 键编辑打分,SingleEdit._compute_edit_logits,edit_logits [B×5 + N],show_source_block: single_edit.py
4,5. 编辑应用,chem.apply_edits_to_mol,Synthon fragments,show_source_block: chem.py
5,6. LG 自回归预测,LGIndEmbed._compute_lg_logits,"lg_scores [B, max_frags, vocab]",show_source_block: lg_ind_embed.py


## 10. 建议的阅读顺序

如果你是第一次系统学习 GraphRetro，建议按以下顺序反复阅读：

1. **先理解数据表示** — 回到 `mol_features.py`，搞清楚 87 维原子特征和 6 维键特征是怎么来的
2. **再理解图构建** — 看 `pack_graph_feats()` 如何把分子变成 batch 张量
3. **理解编码器** — 读 `WLNEncoder.forward()`，理解消息传递的门控机制
4. **理解 Stage 1** — 读 `SingleEdit._compute_edit_logits()`，理解键打分和原子打分
5. **理解 Stage 2** — 读 `LGIndEmbed._compute_lg_logits()`，理解自回归生成
6. **最后** — 读 `apply_edits_to_mol()`，理解化学约束的处理

---

**GraphRetro vs G2Gs 的核心区别：**

| 维度 | G2Gs | GraphRetro |
|------|------|------------|
| Stage 1 | 预测 reaction center (节点/边) | 预测 bond edit (键级变化) |
| Stage 2 | 逐步生成 reactant (beam search) | 从词汇表选择 leaving group |
| 中间表示 | synthon = center 切割产物 | synthon = edit 应用后的碎片 |
| 生成方式 | 图编辑 (逐步添加原子/键) | 分类 (从 LG 词汇表中选择) |

不要追求一遍读完就记住所有细节。对学生来说，更重要的是先建立**两阶段分解**的整体框架：  
**预测在哪里断键 → 预测断出来的碎片需要补什么 → 得到反应物**
